# Notebook 2: Multi-Output XGBoost — GHG Forecasting

Predicts 6 GHG targets simultaneously using a multi-output XGBoost pipeline with LOOCV.

**Targets**: `co2`, `methane`, `nitrous_oxide`, `oil_co2`, `gas_co2`, `coal_co2`

**Features**: `Diesel`, `Kerosene`, `Crude Oil Production`, `Gasoline`, `oil_co2`, `cumulative_ghg`

**Pipeline**: Imputer → StandardScaler → MultiOutputRegressor(XGBoost) with GridSearchCV + LOOCV

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneOut, GridSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

## 2. Load Data & Feature Engineering

The `cumulative_ghg` feature is a composite of cumulative CO2, methane, and nitrous oxide — created to capture the long-term trajectory of total emissions.

In [ ]:
df = pd.read_csv('data/merged_co2_energy_dataset.csv')

# Drop high-correlation derived columns to avoid leakage
exclude_cols = [
    'co2_per_unit_energy', 'co2_growth_prct', 'co2_per_gdp', 'flaring_co2',
    'flaring_co2_per_capita', 'co2_per_capita', 'methane_per_capita', 'primary_energy_consumption'
]
df.drop(columns=exclude_cols, inplace=True, errors='ignore')
df = df.sort_values('year')

# Composite cumulative GHG feature
df['cumulative_ghg'] = (
    df['methane'].cumsum() +
    df['co2'].cumsum() +
    df['nitrous_oxide'].cumsum()
)

features = ['Diesel', 'Kerosene', 'Crude Oil Production', 'Gasoline', 'oil_co2', 'cumulative_ghg']
targets  = ['co2', 'methane', 'nitrous_oxide', 'oil_co2', 'gas_co2', 'coal_co2']

X = df[features]
Y = df[targets]

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

## 3. Evaluation Metrics

In [ ]:
def smape(y_true, y_pred):
    return 100 / len(y_true) * np.sum(
        2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true) + 1e-8)
    )

def mape(y_true, y_pred):
    return 100 * np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8)))

## 4. Model Training — GridSearchCV + LOOCV

LOOCV was chosen over k-fold due to the small dataset size — every sample is used as a test set exactly once, maximising training data at each fold.

In [ ]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
    ('model',   MultiOutputRegressor(
        XGBRegressor(objective='reg:squarederror', random_state=42)
    ))
])

param_grid = {
    'model__estimator__n_estimators':  [100, 200],
    'model__estimator__max_depth':     [3, 5],
    'model__estimator__learning_rate': [0.01, 0.05, 0.1],
    'model__estimator__subsample':     [0.8, 1.0],
    'model__estimator__colsample_bytree': [0.8, 1.0]
}

loo = LeaveOneOut()
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=loo,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X, Y)
best_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

## 5. Evaluation

In [ ]:
Y_pred = best_model.predict(X)

print("[Multi-Output XGBoost — LOOCV Evaluation]")
for i, col in enumerate(targets):
    y_true = Y.iloc[:, i].values
    y_pred = Y_pred[:, i]
    print(f"\n--- {col.upper()} ---")
    print(f"  R²:    {r2_score(y_true, y_pred):.5f}")
    print(f"  MAE:   {mean_absolute_error(y_true, y_pred):.5f}")
    print(f"  MSE:   {mean_squared_error(y_true, y_pred):.5f}")
    print(f"  MAPE:  {mape(y_true, y_pred):.3f}%")
    print(f"  sMAPE: {smape(y_true, y_pred):.3f}%")

## 6. Residual Analysis & Confidence Intervals

In [ ]:
os.makedirs('outputs', exist_ok=True)
residuals = Y_pred - Y.values

fig, axs = plt.subplots(len(targets), 2, figsize=(14, 4 * len(targets)))

for i, target in enumerate(targets):
    std_resid = np.std(residuals[:, i])
    ci = 1.96 * std_resid

    # Predicted vs Actual
    axs[i, 0].scatter(Y[target], Y_pred[:, i], alpha=0.6, color='steelblue', label='Predictions')
    axs[i, 0].plot([Y[target].min(), Y[target].max()],
                   [Y[target].min(), Y[target].max()], 'r--', label='Perfect fit')
    axs[i, 0].set_title(f"{target.upper()} — Predicted vs Actual")
    axs[i, 0].set_xlabel('Actual')
    axs[i, 0].set_ylabel('Predicted')
    axs[i, 0].legend()
    axs[i, 0].grid(True)

    # Residual distribution with 95% CI
    sns.histplot(residuals[:, i], kde=True, stat='density', ax=axs[i, 1], color='skyblue')
    axs[i, 1].axvline(-ci, color='red', linestyle='--', label=f'95% CI: ±{ci:.4f}')
    axs[i, 1].axvline( ci, color='red', linestyle='--')
    axs[i, 1].set_title(f"{target.upper()} — Residual Distribution")
    axs[i, 1].legend()
    axs[i, 1].grid(True)

plt.tight_layout()
plt.savefig('outputs/xgboost_multioutput_residuals.png', dpi=150)
plt.show()

## 7. Save Model

In [ ]:
joblib.dump(best_model, 'outputs/xgb_multioutput_model.pkl')
print('Model saved to outputs/xgb_multioutput_model.pkl')